In [ ]:
#!pip install -r ../requirements.txt
#!pip install pmlb lightgbm
#!python -m pip install -e ..


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from pmlb import classification_dataset_names, fetch_data
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from arborration import (
    AADForestGenerativeClassifier,
    IsoForestGenerativeClassifier,
    PineForestGenerativeClassifier,
)

# -------------------------------------------------------------------
# 1) Dataset selection helpers
# -------------------------------------------------------------------

def collect_classification_metadata(max_samples_cap=100_000, max_feats_cap=100):
    """Collect basic metadata for PMLB classification datasets."""
    meta = []
    for name in classification_dataset_names:
        try:
            X, y = fetch_data(name, return_X_y=True)
        except Exception as e:
            print(f"[WARN] Failed to load dataset {name}: {e}")
            continue

        n_samples, n_features = X.shape
        if n_samples < 2:
            continue

        if n_samples > max_samples_cap or n_features > max_feats_cap:
            continue

        n_classes = np.unique(y).shape[0]
        if n_classes < 2:
            continue

        meta.append(
            {
                "name": name,
                "n_samples": n_samples,
                "n_features": n_features,
                "n_classes": n_classes,
            }
        )

    return pd.DataFrame(meta)


def split_into_size_buckets(
    meta_df,
    small_max=1_000,
    med_max=10_000,
    n_small=20,
    n_med=20,
    n_large=20,
    random_state=0,
):
    """Split datasets into small/medium/large buckets and sample each bucket."""
    rng = np.random.RandomState(random_state)

    small = meta_df[meta_df["n_samples"] <= small_max].copy()
    medium = meta_df[
        (meta_df["n_samples"] > small_max) & (meta_df["n_samples"] <= med_max)
    ].copy()
    large = meta_df[meta_df["n_samples"] > med_max].copy()

    def pick(df, n):
        if len(df) <= n:
            return df
        idx = np.arange(len(df))
        rng.shuffle(idx)
        return df.iloc[idx[:n]]

    selected = pd.concat(
        [
            pick(small, n_small).assign(size_bucket="small"),
            pick(medium, n_med).assign(size_bucket="medium"),
            pick(large, n_large).assign(size_bucket="large"),
        ],
        ignore_index=True,
    )
    return selected


# -------------------------------------------------------------------
# 2) AUC helper that handles binary vs multiclass
# -------------------------------------------------------------------

def compute_roc_auc(y_true, proba):
    """Compute ROC AUC for binary or multiclass probability outputs."""
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)

    n_classes = proba.shape[1] if proba.ndim == 2 else 2
    if n_classes == 2:
        return roc_auc_score(y_true, proba[:, 1])

    return roc_auc_score(y_true, proba, multi_class="ovr", average="macro")


# -------------------------------------------------------------------
# 3) Benchmark routine
# -------------------------------------------------------------------

def build_models(n_classes, forest_params=None):
    if forest_params is None:
        forest_params = {
            "n_estimators": 100,
            "max_samples": 256,
            "n_jobs": -1,
            "random_state": 0,
            "verbose": 0,
            "max_depth": None,
            "calibration": "discriminator",
            "global_scale": 1.0,
        }

    models = {
        "GaussianNB": GaussianNB(),
        "LogisticRegression": make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=1000,
                n_jobs=-1,
                solver="lbfgs",
                multi_class="auto",
                random_state=0,
            ),
        ),
        "IsoForestGenerativeClassifier": IsoForestGenerativeClassifier(**forest_params),
        "PineForestGenerativeClassifier": PineForestGenerativeClassifier(**forest_params),
        "AADForestGenerativeClassifier": AADForestGenerativeClassifier(**forest_params),
    }

    if n_classes == 2:
        models["LGBMClassifier"] = LGBMClassifier(
            n_estimators=200,
            learning_rate=0.1,
            n_jobs=-1,
            random_state=0,
            verbose=-1,
        )
    else:
        models["LGBMClassifier"] = LGBMClassifier(
            n_estimators=200,
            learning_rate=0.1,
            objective="multiclass",
            num_class=n_classes,
            n_jobs=-1,
            random_state=0,
            verbose=-1,
        )

    return models


def benchmark_models_on_datasets(
    dataset_meta,
    test_size=0.3,
    random_state=42,
    forest_params=None,
    verbose=True,
):
    results = []

    for _, row in dataset_meta.iterrows():
        name = row["name"]
        size_bucket = row["size_bucket"]
        n_samples = row["n_samples"]
        n_features = row["n_features"]
        n_classes = row["n_classes"]

        if verbose:
            print(
                f"\n=== Dataset: {name} | bucket={size_bucket} | samples={n_samples} | "
                f"features={n_features} | classes={n_classes} ==="
            )

        try:
            X, y = fetch_data(name, return_X_y=True)
        except Exception as e:
            print(f"[WARN] Skipping dataset {name}, failed to fetch: {e}")
            continue

        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X,
                y,
                test_size=test_size,
                random_state=random_state,
                stratify=y,
            )
        except Exception as e:
            print(f"[WARN] Skipping dataset {name}, train_test_split failed: {e}")
            continue

        models = build_models(n_classes=n_classes, forest_params=forest_params)

        for model_name, model in models.items():
            train_time = np.nan
            infer_time = np.nan
            auc = np.nan

            try:
                t0 = time.perf_counter()
                model.fit(X_train, y_train)
                train_time = time.perf_counter() - t0

                t0 = time.perf_counter()
                proba = model.predict_proba(X_test)
                infer_time = time.perf_counter() - t0

                auc = compute_roc_auc(y_test, proba)
                if verbose:
                    print(f"  {model_name}: roc_auc={auc:.4f}")
            except Exception as e:
                print(f"[WARN] {model_name} failed on {name}: {e}")

            results.append(
                {
                    "dataset": name,
                    "size_bucket": size_bucket,
                    "n_samples": n_samples,
                    "n_features": n_features,
                    "n_classes": n_classes,
                    "model": model_name,
                    "train_time": train_time,
                    "infer_time": infer_time,
                    "roc_auc": auc,
                }
            )

    return pd.DataFrame(results)


# -------------------------------------------------------------------
# 4) Run everything
# -------------------------------------------------------------------

meta_df = collect_classification_metadata(max_samples_cap=100_000)
print(f"Collected metadata for {len(meta_df)} classification datasets.")

selected = split_into_size_buckets(
    meta_df,
    small_max=1_000,
    med_max=10_000,
    n_small=12,
    n_med=12,
    n_large=12,
    random_state=0,
)

print(selected[["name", "size_bucket", "n_samples", "n_features", "n_classes"]])

results_df = benchmark_models_on_datasets(
    selected,
    test_size=0.3,
    random_state=42,
    forest_params=None,
    verbose=True,
)

output_dir = Path("results")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "forest_generators_vs_baselines_pmlb.csv"
results_df.to_csv(output_path, index=False)
print(f"Saved benchmark results to {output_path}")
results_df.head()



=== Dataset: dermatology | bucket=small | samples=366 | features=34 | classes=6 ===
0.9765527310006715
0.9781901579012563
0.9987106958216798


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.9964144391001059

=== Dataset: soybean | bucket=small | samples=675 | features=35 | classes=18 ===
0.9912896729960222
0.9886317142253753
0.9933940874999211


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.9974600540165363

=== Dataset: iris | bucket=small | samples=150 | features=4 | classes=3 ===
0.997037037037037
0.997037037037037
0.9985185185185186
0.9918518518518519

=== Dataset: hepatitis | bucket=small | samples=155 | features=19 | classes=2 ===


...

0.9994200277835812

=== Dataset: shuttle | bucket=large | samples=58000 | features=9 | classes=7 ===
0.9669463190238412
0.981402314185979


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


0.9581764837624368


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.6532160867042233

Benchmark results (head):
       dataset size_bucket  n_samples  n_features  n_classes  \
0  dermatology       small        366          34          6   
1  dermatology       small        366          34          6   
2  dermatology       small        366          34          6   
3  dermatology       small        366          34          6   
4      soybean       small        675          35         18   

                model  train_time  infer_time   roc_auc  
0                 HRP    0.240293    0.005745  0.976553  
1          NaiveBayes    0.012194    0.017239  0.978190  
2  LogisticRegression    0.056322    0.005128  0.998711  
3      LGBMClassifier    1.270116    0.013645  0.996414  
4                 HRP    0.232963    0.015370  0.991290  


In [ ]:
#results_df=pd.read_csv('/content/iforest_gen_vs_lr_lgbm_pmlb_benchmark_naive.csv')
non_iso_df=results_df[results_df['model']!='HRP'].copy()

In [ ]:
results_df=pd.concat([non_iso_df,results_df])

In [ ]:
non_iso_df.iloc[:30]

In [ ]:
results_df

In [ ]:
results_df['time_rank']=results_df.groupby('dataset')['train_time'].rank()
results_df.groupby('model')['time_rank'].mean()

,time_rank
model,
HRP,2.15625
LGBMClassifier,3.90625
LogisticRegression,2.93750
NaiveBayes,1.00000


In [ ]:
results_df['time_rank']=results_df.groupby('dataset')['roc_auc'].rank()
results_df.groupby('model')['time_rank'].mean()

,time_rank
model,
HRP,2.515625
LGBMClassifier,3.156250
LogisticRegression,2.656250
NaiveBayes,1.671875


In [ ]:
results_df.loc[results_df.groupby('dataset')['roc_auc'].idxmax()]['model'].value_counts()

,count
model,
LGBMClassifier,20
LogisticRegression,7
HRP,3
NaiveBayes,2


In [ ]:
results_df.loc[results_df.groupby('dataset')['roc_auc'].idxmax()]

,dataset,size_bucket,n_samples,n_features,n_classes,model,train_time,infer_time,roc_auc,time_rank
63,GAMETES_Epistasis_3_Way_20atts_0.2H_EDM_1_1,medium,1600,20,2,LGBMClassifier,0.101287,0.006996,0.613898,4.0
99,adult,large,48842,14,2,LGBMClassifier,0.704878,0.132012,0.929743,4.0
79,agaricus_lepiota,medium,8145,22,2,LGBMClassifier,0.229537,0.017614,1.000000,4.0
91,allhyper,medium,3771,29,4,LGBMClassifier,0.413583,0.040854,0.998913,4.0
55,allhypo,medium,3770,29,3,LGBMClassifier,2.032330,0.058380,0.998070,4.0
27,cars,small,392,8,3,LGBMClassifier,0.096880,0.005127,0.986553,4.0
82,coil2000,medium,9822,85,2,LogisticRegression,1.105015,0.003937,0.702200,4.0
103,connect_4,large,67557,42,3,LGBMClassifier,2.532586,0.577920,0.921007,4.0
39,corral,small,160,6,2,LGBMClassifier,0.015521,0.001653,1.000000,4.0
2,dermatology,small,366,34,6,LogisticRegression,0.056322,0.005128,0.998711,4.0
